In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

SEED = 42
TEST_SIZE = 0.2
DATA = "../data/train.csv"
DROP = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp"]

df = pd.read_csv(DATA).drop(columns=DROP)
X, y = df.drop(columns="Survived"), df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

In [ ]:
num_cols = ["Age", "Fare", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh",  OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

model = Pipeline([
    ("prep", preprocess),
    ("rf",   RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        min_samples_leaf=1,
        oob_score=True,
        n_jobs=-1,
        random_state=SEED,
    )),
])
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("Test AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("OOB score:", round(model.named_steps["rf"].oob_score_, 4))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0      0.818     0.900     0.857       110
           1      0.810     0.681     0.740        69

    accuracy                          0.816       179
   macro avg      0.814     0.791     0.799       179
weighted avg      0.815     0.816     0.812       179

Test AUC: 0.8368
OOB score: 0.8062
[[99 11]
 [22 47]]


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
from utils import log_result

log_result(
    model_name="Random Forest",
    y_true=y_test,
    y_proba=y_proba,      
    y_pred=y_pred,       
    params={"n_estimators": "500", "max_features": "sqrt"},
    notes=""
)

✓ Random Forest: AUC=0.8368, acc=0.8156 → c:\Users\АуфПК\VSCode\titanic\results.csv


{'model': 'Random Forest',
 'timestamp': '2026-11-03 00:29',
 'auc': 0.8368,
 'accuracy': 0.8156,
 'f1': 0.7402,
 'precision': 0.8103,
 'recall': 0.6812,
 'params': '{}',
 'notes': ''}